# C1 | Figure 2

Saved SHINE results. Training seed 123; K=11; clustering seed 0.

Five-fit summaries use seeds 0, 1, 2, 123 and 1234. See [setup and data requirements](README.md).

Inputs: pre-registered SMA data released with [NicheTrans](https://github.com/YSTLab/NicheTrans), restricted to the frozen common paired cohort. Registration precedes SHINE training.


In [25]:
import os
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import shine
from shine import SHINE, normalize_counts
from shine.study import Section

CASE = "C1"
# Extract the figure-data archive into data/, or set SHINE_DATA_ROOT.
search_roots = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("SHINE_DATA_ROOT"):
    ROOT = Path(os.environ["SHINE_DATA_ROOT"]).expanduser().resolve()
else:
    ROOT = next((candidate for parent in search_roots
                 for candidate in (parent / "data", parent)
                 if (candidate / "revision_experiments").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the figure-data archive into data/, or set SHINE_DATA_ROOT. See data/README.md.")
manifest_candidates = [p / name for p in search_roots
                       for name in ("sections.json", "notebooks/sections.json")]
manifest_candidates.append(ROOT / "SHINE-github/notebooks/sections.json")
MANIFEST = next((p for p in manifest_candidates if p.is_file()), None)
if MANIFEST is None:
    raise FileNotFoundError("Cannot find notebooks/sections.json.")
import sys
sys.path.insert(0, str(MANIFEST.parent))
from figure_plotting import spatial_results, regional_results, molecular_maps, attention_plot, save_panel

OUT = ROOT / "outputs/notebook_results_release_011_20261127" / CASE
OUT.mkdir(parents=True, exist_ok=True)
section = Section(ROOT, MANIFEST, CASE)
print("shine-omics", shine.__version__, "|", CASE, "|", len(section.spots), "paired spots")

shine-omics 0.1.1 | C1 | 2378 paired spots


## 1. Settings

Fixed figure settings and paired cohort. Source files are verified by SHA-256.

In [26]:
settings = {"Figure": section.config["figure"], "K": section.config["K"],
            "epochs": 120, "training_seed": 123,
            "clustering_seed": section.config["display_clustering_seed"],
            "local_mask": True, "local_neighbors": section.config["graph_options"]["local_k"],
            "spatial_graph_neighbors": section.config["graph_options"]["spatial_k"], "final_ReLU": True, **section.config["model_options"]}
display(pd.Series(settings, name="Final setting").to_frame())
display(section.spots.groupby(["region", "lesion"], dropna=False).size().rename("paired_spots").reset_index())

                           Final setting
Figure                                 2
K                                     11
epochs                               120
training_seed                        123
clustering_seed                        0
local_mask                          True
local_neighbors                        4
spatial_graph_neighbors                6
final_ReLU                          True
loss                            original
l2                                 False
spatial_floor                        0.0
latent_dim                           128
attention_backend        grouped_feature
evaluate_zero_alignment            False

   region    lesion  paired_spots
0     ACB    intact           127
1     ACB  lesioned           121
2      CP    intact           340
3      CP  lesioned           320
4     CTX    intact           497
5     CTX  lesioned           314
6     LSX    intact            46
7     LSX  lesioned            76
8     MSC    intact            23
9     MSC  lesioned            85
10     OT    intact            30
11     OT  lesioned            44
12     VL    intact            33
13     VL  lesioned            36
14    aco    intact             8
15    aco  lesioned             7
16     cc    intact            84
17     cc  lesioned            82
18    NaN    intact            67
19    NaN  lesioned            38

## 2. Inputs

These paired matrices are already preprocessed. Do not repeat normalization or registration.

In [27]:
print(section.config["preprocessing"])
input_audit = pd.DataFrame([
    {"modality": name, "spots": x.shape[0], "features": x.shape[1],
     "finite": bool(np.isfinite(x).all()), "constant_rows": int((x.std(1) == 0).sum()),
     "minimum": float(x.min()), "maximum": float(x.max())}
    for name, x in [("RNA", section.rna), ("MSI", section.msi)]])
display(input_audit)
assert section.spots.spot.is_unique
assert np.allclose(section.xy, section.spots[["x", "y"]])
# For raw counts: normalize_counts(full_counts, feature_indices=frozen_columns)
# Do not renormalize section.rna or section.msi.

Frozen NicheTrans paired cohort. RNA: all-gene total 10000/log1p, frozen paper B1 2000 genes. MSI: TIC 10000/log1p, frozen paired assignment. Do not normalize inputs.npz again.


  modality  spots  features  finite  constant_rows  minimum   maximum
0      RNA   2378      2000    True              0      0.0  6.241874
1      MSI   2378      1538    True              0      0.0  7.090251

## 3. Graphs

H5 + H20; spatial kNN6; feature correlation-kNN20; local self+4 attention. Graphs use no annotation labels.

In [28]:
data, graphs = section.prepare()
display(pd.Series(graphs["audit"], name="Graph audit").to_frame())
print("Validated tensor shape:", data.shape)
print("Input + graph fingerprint:", data.fingerprint())

Validated tensor shape: (2378, 2000, 1538)
Input + graph fingerprint: e2cd5168cbe96909531c6f1517a2fd72655b23a1f4b3ff178e73afd555d3fd81


                                     Graph audit
n_spots                                     2378
spatial_k                                      6
feature_k                                     20
local_mask                                  True
local_k                                        4
cross_modal_support          self_plus_neighbors
hyperedge_sizes                          [5, 20]
n_hyperedges                                4756
pca_components                                30
pca_seed                                       0
constant_feature_rows                     [0, 0]
allow_constant_feature_rows                 True
labels_used                                False
sparse_propagation                         False

## 4. Model

Replay the saved figure model. Set `RUN_NEW_TRAINING=True` for a separate 120-epoch fit.

In [29]:
model, labels, replay_audit = section.replay(data)
display(pd.Series(replay_audit).to_frame("Replay check"))
RUN_NEW_TRAINING = False
if RUN_NEW_TRAINING:
    new_model = SHINE(**section.config["model_options"])
    def progress(record):
        if record["epoch"] % 20 == 0:
            print("Epoch", record["epoch"], "loss", round(record["total"], 5))
    new_model.fit(data, epochs=120, seed=123, lr=1e-4, weight_decay=1e-3, progress=progress)
    new_labels = new_model.cluster(section.config["K"], seed=section.config["display_clustering_seed"])
    new_model.save(OUT / "new_training_seed123.pt")
    display(section.score(new_labels))
else:
    print("Displaying the saved figure model.")

Displaying the saved figure model.


                                                           Replay check
embedding_max_abs_error                                             0.0
figure_labels_exact                                                True
n_spots                                                            2378
origin                   saved figure checkpoint; new package inference

## 5. ARI and NMI

Average five clustering seeds within each fit, then report mean and sample SD over five training seeds. Regional scores retain whole-section cluster IDs.

In [30]:
display_metrics = section.score(labels)
raw_metrics, per_fit, summary = section.five_fit_metrics()
print("Displayed fit: seed 123")
display(display_metrics.round(4))
print("Five fits: mean and sample SD")
display(summary.round(4))
display(per_fit.round(4))
for name, table in [("display_metrics", display_metrics), ("all_partitions", raw_metrics),
                    ("per_training_fit", per_fit), ("five_fit_summary", summary)]:
    table.to_csv(OUT / (name + ".csv"), index=False)

Displayed fit: seed 123
Five fits: mean and sample SD


        endpoint  n_scored     ARI     NMI
0  Whole section      2273  0.4913  0.6095
1             CP       660  0.8550  0.7661

        endpoint  ARI_mean  ARI_std  NMI_mean  NMI_std  n_fits
0             CP    0.8198   0.0257    0.7320   0.0257       5
1  Whole section    0.5196   0.0305    0.6053   0.0062       5

        endpoint  training_seed     ARI     NMI
0             CP              0  0.8014  0.7185
1             CP              1  0.8435  0.7545
2             CP              2  0.7919  0.7013
3             CP            123  0.8501  0.7624
4             CP           1234  0.8123  0.7231
5  Whole section              0  0.5725  0.6093
6  Whole section              1  0.5149  0.5962
7  Whole section              2  0.5105  0.6015
8  Whole section            123  0.5055  0.6098
9  Whole section           1234  0.4945  0.6100

## 6. Spatial partition and UMAP

Manuscript colors and spatial rendering; seed 123. Filled display cells remove scatter gaps without changing measured spots or cluster IDs. Grey denotes unlabelled or unpaired tissue. UMAP is used only for visualization.


In [31]:
import umap
saved_embedding = np.load(section.path("embedding_123"))
umap_xy = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=123, n_jobs=1).fit_transform(saved_embedding)
fig = spatial_results(section, labels, umap_xy)
save_panel(fig, OUT / "seed123_spatial_UMAP")
plt.show()
spatial_source = section.spots.copy()
spatial_source["SHINE_cluster"] = labels
spatial_source[["UMAP1", "UMAP2"]] = umap_xy
spatial_source.to_csv(OUT / "seed123_spatial_source.csv", index=False)

<Figure size 1050x410 with 3 Axes>

In [32]:
fig = regional_results(section, labels)
save_panel(fig, OUT / "seed123_regions")
plt.show()

<Figure size 400x350 with 1 Axes>

## 7. Attention

Spatial-branch weights within RNA and MSI. These are mixing weights, not causal importance.

In [33]:
fig = attention_plot(model)
save_panel(fig, OUT / "attention_weights")
plt.show()
attention_source = pd.DataFrame({"spot": section.spots.spot,
    "RNA_spatial_raw": model.attention_["rna_raw"][:, 0],
    "RNA_spatial_effective": model.attention_["rna"][:, 0],
    "MSI_spatial_raw": model.attention_["msi_raw"][:, 0],
    "MSI_spatial_effective": model.attention_["msi"][:, 0]})
attention_source.to_csv(OUT / "attention_source.csv", index=False)
display(attention_source.drop(columns="spot").describe().round(3))

<Figure size 720x260 with 2 Axes>

       RNA_spatial_raw  ...  MSI_spatial_effective
count         2378.000  ...               2378.000
mean             0.018  ...                  0.004
std              0.008  ...                  0.001
min              0.009  ...                  0.002
25%              0.014  ...                  0.004
50%              0.016  ...                  0.004
75%              0.019  ...                  0.004
max              0.197  ...                  0.007

[8 rows x 4 columns]

## 8. Measured genes and MSI peaks

Shared Figure 2 markers. Anatomical-side effects and cluster comparisons are reported separately.

In [34]:
molecules = section.molecular_values()
print("RNA:", section.config["display_genes"])
print("MSI m/z:", section.config["display_peaks"])
fig = molecular_maps(section, molecules)
save_panel(fig, OUT / "measured_molecules")
plt.show()
measured_source = pd.concat([section.spots[["spot", "x", "y", "region", "lesion"]], molecules], axis=1)
measured_source.to_csv(OUT / "measured_molecule_source.csv", index=False)
effects = section.molecular_comparison(molecules, labels)
display(effects.round(3))
effects.to_csv(OUT / "measured_molecule_effects.csv", index=False)

RNA: ['Penk', 'Pcp4', 'Tac1', 'Sfxn1', 'Cst3', 'Cartpt']
MSI m/z: ['674.28833', '390.16864', '674.28592', '435.20692', '675.29202', '674.28050']


<Figure size 1200x450 with 13 Axes>

                     contrast    feature  ...  mean_target  difference_SD
0   CP: lesioned minus intact       Penk  ...        3.637          0.960
1   CP: lesioned minus intact       Pcp4  ...        2.723         -0.674
2   CP: lesioned minus intact       Tac1  ...        0.849         -0.849
3   CP: lesioned minus intact      Sfxn1  ...        0.203         -0.423
4   CP: lesioned minus intact       Cst3  ...        3.820          0.522
5   CP: lesioned minus intact     Cartpt  ...        0.384          0.635
6   CP: lesioned minus intact  674.28833  ...        0.104         -1.887
7   CP: lesioned minus intact  390.16864  ...        0.208         -1.934
8   CP: lesioned minus intact  674.28592  ...        0.123         -1.849
9   CP: lesioned minus intact  435.20692  ...        0.052         -1.938
10  CP: lesioned minus intact  675.29202  ...        0.092         -1.834
11  CP: lesioned minus intact  674.28050  ...        0.103         -1.745

[12 rows x 7 columns]

## 9. CP clusters and pathways

Frozen Figure 2 results. Metascape used the top 100 genes per cluster and a mouse-genome background.

In [35]:
section.verify("molecular_selection")
cluster_molecules = pd.read_csv(section.path("molecular_selection"))
cluster_molecules = cluster_molecules[cluster_molecules.section.eq(CASE)]
display(cluster_molecules[["side", "cluster", "feature", "modality", "cluster_mean", "centered_mean_SD"]].round(3))
section.verify("enrichment")
enrichment = pd.read_csv(section.path("enrichment"))
display(enrichment[enrichment.section.eq(CASE)])

        side  cluster    feature modality  cluster_mean  centered_mean_SD
48    intact        1       Penk      RNA         2.855            -0.462
49  lesioned        8       Penk      RNA         3.732             0.610
50    intact        1       Pcp4      RNA         3.199             0.371
51  lesioned        8       Pcp4      RNA         2.761            -0.290
52    intact        1       Tac1      RNA         1.549             0.406
53  lesioned        8       Tac1      RNA         0.894            -0.383
54    intact        1     Cartpt      RNA         0.043            -0.314
55  lesioned        8     Cartpt      RNA         0.419             0.394
56    intact        1      Sfxn1      RNA         0.431             0.208
57  lesioned        8      Sfxn1      RNA         0.210            -0.204
58    intact        1       Cst3      RNA         3.600            -0.258
59  lesioned        8       Cst3      RNA         3.815             0.256
60    intact        1  674.28833      

               GO  ... gene_ratio
4        mmu04728  ...   0.062500
5        mmu04728  ...        NaN
10     GO:0014059  ...   0.052083
11     GO:0014059  ...        NaN
16     GO:0050433  ...   0.052083
17     GO:0050433  ...        NaN
22         WP2910  ...        NaN
23         WP2910  ...        NaN
28     GO:0098916  ...   0.135417
29     GO:0098916  ...        NaN
34     GO:0007268  ...   0.135417
35     GO:0007268  ...        NaN
40     GO:0042063  ...   0.072917
41     GO:0042063  ...        NaN
46     GO:0042552  ...   0.072917
47     GO:0042552  ...        NaN
52     GO:0048709  ...   0.031250
53     GO:0048709  ...        NaN
58     GO:0002696  ...        NaN
59     GO:0002696  ...        NaN
64  R-MMU-2132295  ...        NaN
65  R-MMU-2132295  ...        NaN
70       mmu05012  ...        NaN
71       mmu05012  ...   0.051020
76       mmu05022  ...   0.135417
77       mmu05022  ...        NaN
82     GO:1903351  ...        NaN
83     GO:1903351  ...        NaN

[28 rows x 15

## 10. Validation and exports

Saved-model replay and fresh training are reported separately. Fresh fits do not replace the figure model.

In [36]:
print("Saved weights replayed against the updated four-neighbor figure. No new training is run by default.")
provenance = {"case": CASE, "package_version": shine.__version__, "settings": settings,
              "replay": replay_audit, "sources": section.config["sources"],
              "statistical_unit": "training seed; sample SD over five fits",
              "spatial_train_seed": 123, "new_training_replaces_figure": False}
(OUT / "notebook_provenance.json").write_text(json.dumps(provenance, indent=2))
print("Output directory:", OUT)

Saved weights replayed against the updated four-neighbor figure. No new training is run by default.
Output directory: /Users/dubingxue/Desktop/SHINE/outputs/notebook_results_release_011_20261127/C1
